# Phase 3 — Training Job

**Before running this notebook:**
1. Test `src/train.py` locally — `python src/train.py --training_data ../data/processed/train_clean.csv`
2. Fix any errors locally. Only submit the job once it runs clean.
3. Phase 2 gate must be passed (compute + env + data asset exist).

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
# Same values as Phase 2 — just change PROJECT_NAME.
PROJECT_NAME    = 'my-project'

DATA_ASSET_NAME = f'{PROJECT_NAME}-data'
COMPUTE_NAME    = 'aml-cluster'
ENV_NAME        = f'{PROJECT_NAME}-env'
EXPERIMENT_NAME = f'{PROJECT_NAME}-training'

In [ ]:
# ── IMPORTS ───────────────────────────────────────────────────────────────────
from azure.ai.ml import MLClient, Input, command
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential

## Section 1 — Connect

In [ ]:
# ── AUTH ──────────────────────────────────────────────────────────────────────
try:
    credential = DefaultAzureCredential()
    credential.get_token('https://management.azure.com/.default')
except Exception:
    credential = InteractiveBrowserCredential()

ml_client = MLClient.from_config(credential=credential)
print(f'✓ Connected to: {ml_client.workspace_name}')

## Section 2 — Write Training Script

> The cell below writes `src/train.py` from this notebook.  
> Edit the functions directly here, then run the cell. This keeps the script in sync with what you test locally.

In [ ]:
%%writefile ../src/train.py
import argparse
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score


# ── GET DATA ──────────────────────────────────────────────────────────────────
def get_data(path):
    df = pd.read_csv(path)
    return df


# ── SPLIT DATA ────────────────────────────────────────────────────────────────
def split_data(df):
    # Replace 'label' with your TARGET column name
    X = df.drop(columns=['label']).values
    y = df['label'].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=0
    )
    return X_train, X_test, y_train, y_test


# ── TRAIN MODEL ───────────────────────────────────────────────────────────────
def train_model(X_train, y_train, reg_rate):
    model = LogisticRegression(C=1 / reg_rate, solver='lbfgs', max_iter=1000)
    model.fit(X_train, y_train)
    return model


# ── EVAL MODEL ────────────────────────────────────────────────────────────────
def eval_model(model, X_test, y_test):
    y_hat = model.predict(X_test)
    y_scores = model.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_hat)
    auc = roc_auc_score(y_test, y_scores)
    return acc, auc


# ── MAIN ──────────────────────────────────────────────────────────────────────
def main(args):
    mlflow.sklearn.autolog()

    df = get_data(args.training_data)
    X_train, X_test, y_train, y_test = split_data(df)
    model = train_model(X_train, y_train, args.reg_rate)
    acc, auc = eval_model(model, X_test, y_test)

    mlflow.log_metric('val_acc', acc)
    mlflow.log_metric('val_auc', auc)

    print(f'Accuracy : {acc:.4f}')
    print(f'AUC      : {auc:.4f}')


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--training_data', type=str, required=True)
    parser.add_argument('--reg_rate',      type=float, default=0.01)
    args = parser.parse_args()

    main(args)

## Section 3 — Submit Command Job

In [ ]:
# ── COMMAND JOB ───────────────────────────────────────────────────────────────
# inputs.${{...}} syntax lets Azure resolve the data asset path at runtime.
job = command(
    code='../src',
    command='python train.py --training_data ${{inputs.training_data}} --reg_rate ${{inputs.reg_rate}}',
    inputs={
        'training_data': Input(
            type=AssetTypes.URI_FILE,
            path=f'azureml:{DATA_ASSET_NAME}@latest',
        ),
        'reg_rate': 0.01,
    },
    environment=f'{ENV_NAME}@latest',
    compute=COMPUTE_NAME,
    experiment_name=EXPERIMENT_NAME,
    display_name='training-run-v1',
)

returned_job = ml_client.jobs.create_or_update(job)

print(f'✓ Job submitted: {returned_job.name}')
print(f'  Status      : {returned_job.status}')
print(f'  Studio URL  : {returned_job.studio_url}')

In [ ]:
# ── WAIT FOR COMPLETION ───────────────────────────────────────────────────────
# Optional: stream output in the notebook. Comment out if you prefer to watch in Studio.
ml_client.jobs.stream(returned_job.name)

In [ ]:
# ── CHECK METRICS ─────────────────────────────────────────────────────────────
finished_job = ml_client.jobs.get(returned_job.name)

print(f'Status : {finished_job.status}')
print(f'Job    : {finished_job.name}')
print()
print('→ Copy the job name above — you will need it in Phase 5 (register).')

---
## Phase 3 Gate

- [ ] `finished_job.status == 'Completed'`
- [ ] Metrics (`val_acc`, `val_auc`) visible in Studio → Jobs → [job name] → Metrics
- [ ] Job name recorded in README_TEMPLATE.md → Model Version Log

Next step: Phase 4 (sweep) **or** skip to Phase 5 if reg_rate=0.01 is good enough.